In [ ]:
#| hide

# Annex

> WSL USB passthrough, naming conventions, package.xml/setup.py, and common pitfalls.

## WSL USB Passthrough

By default, WSL2 cannot access USB devices. You need `usbipd-win` to attach them.

### Install usbipd-win

On **Windows** (PowerShell as Admin):

```powershell
winget install usbipd
```

### Attach a USB camera

```powershell
# List USB devices
usbipd list

# Bind the device (only needed once)
usbipd bind --busid <BUSID>

# Attach to WSL
usbipd attach --wsl --busid <BUSID>
```

In WSL, verify the device is visible:

```sh
$ ls /dev/video*
/dev/video0  /dev/video1
```

### Detach when done

```powershell
usbipd detach --busid <BUSID>
```

> **Gotcha:** If the camera shows up as `/dev/video0` and `/dev/video1`, try both. Often `video0` is the metadata device and `video1` is the actual video stream.

## Naming Conventions

### Notebooks

| Pattern | Example | Description |
|---------|---------|-------------|
| `NN_name.ipynb` | `02_standard_publisher.ipynb` | Two-digit prefix for ordering |
| `#| default_exp <module>` | `#| default_exp cam_pub` | Must match `ros2 run <pkg> <node>` |

### ROS2 Nodes

The `default_exp` in the notebook determines the Python filename, which must match the `ros2 run` command:

| Notebook | default_exp | Run command |
|----------|-------------|-------------|
| `02_standard_publisher.ipynb` | `cam_pub` | `ros2 run ros2_cam_nbdev cam_pub` |
| `03_parameters_and_services.ipynb` | `cfg_node` | `ros2 run ros2_cam_nbdev cfg_node` |
| `04_lifecycle_node.ipynb` | `lc_node` | `ros2 run ros2_cam_nbdev lc_node` |
| `06_calibration_and_info.ipynb` | `calib_node` | `ros2 run ros2_cam_nbdev calib_node` |

> Every node **MUST** define `def main(args=None):` — `universal_setup.py` auto-discovers entry points by scanning for this pattern.

## package.xml and setup.py

These files are **auto-generated** by `bash scripts/compile_pkg.sh`. Do not edit them directly.

### How it works

1. `compile_pkg.sh` runs `universal_setup.py` which scans `ros2_cam_nbdev/*.py` for `def main(args=None):`
2. Each match becomes an entry point in `setup.py`
3. `colcon build` registers the nodes with ROS2

### If you must edit

- Edit the template in `scripts/python_scripts/universal_setup.py`
- Or edit `pyproject.toml` for Python dependencies
- Then re-run `bash scripts/compile_pkg.sh`

## Common Pitfalls

### 1. `ModuleNotFoundError: No module named 'rclpy'`

The `.venv` was created without `--system-site-packages`. Fix:

```sh
$ source /opt/ros/$ROS_DISTRO/setup.bash
$ rm -rf .venv
$ uv venv --python /usr/bin/python3 --system-site-packages
$ uv sync
```

### 2. Camera not detected in WSL

- Make sure `usbipd attach --wsl` succeeded
- Check `ls /dev/video*` in WSL
- Try different `video_device` indices (0, 1, 2...)
- Restart WSL: `wsl --shutdown` then reopen

### 3. Node not found after build

```sh
# Source the workspace overlay
$ source install/setup.bash

# Verify the node exists
$ ros2 pkg executables ros2_cam_nbdev
```

### 4. Images appear black or green

- Wrong pixel format — try `MJPG` or `YUYV`
- Wrong resolution — camera may not support the requested resolution
- Check `v4l2-ctl --list-formats-ext -d /dev/video0` for supported formats

### 5. `cv_bridge` import fails

`cv_bridge` requires OpenCV. Make sure it's installed:

```sh
$ sudo apt install ros-$ROS_DISTRO-cv-bridge
```

If using a custom OpenCV build, you may need to rebuild `cv_bridge` against it.

### 6. High CPU usage

- Reduce FPS: `frame_rate:=15.0`
- Reduce resolution: `resolution_width:=320 resolution_height:=240`
- Use MJPG compression: `video_fourcc:=MJPG`
- Use `image_transport` for compressed topics

### 7. `nbdev-export` doesn't create the .py file

Make sure the notebook has:

- `#| default_exp <module_name>` as the first code cell
- `#| export` on every cell you want exported
- `#| hide
import nbdev; nbdev.nbdev_export()` as the last code cell

## Dependency Management

- Use `uv add <package>` — **never `pip install`**
- For packages that conflict with ROS2 system packages:

```sh
$ uv add <package> --constraint system_constraints.txt
```

- `matplotlib-inline==0.1.6` is **PINNED** — do not upgrade. ROS2 compatibility depends on this exact version.

## Pre-Commit Ritual

Before every commit that touches notebooks or exported code:

```sh
$ uv run nbdev-prepare
```

This runs:

1. `nbdev-export` — notebooks → .py modules
2. `nbdev-test` — run all notebook tests
3. `nbdev-clean` — strip notebook metadata
4. `nbdev-readme` — regenerate README.md from `index.ipynb`

> **This is mandatory.** There is no CI yet — all validation is manual.